In [ ]:
%pip install pandas matplotlib seaborn pyarrow
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Vérification et création forcée du dossier pour les images
os.makedirs('../docs/images', exist_ok=True)

# 2. Chemin du fichier de tes résultats MapReduce
json_path = '../results/rentabilite_horaire.json'

if not os.path.exists(json_path):
    print(f"❌ Erreur : Le fichier {json_path} n'existe pas. Vérifie s'il est bien généré.")
else:
    # Lecture sécurisée du JSON
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Conversion adaptative en DataFrame
    if isinstance(data, dict):
        # Si Spark a écrit le JSON sous forme de dictionnaire clé:objet
        df = pd.DataFrame.from_dict(data, orient='index')
        if 'heure' not in df.columns:
            df['heure'] = df.index.astype(int)
    else:
        # Si c'est une liste classique d'objets JSON
        df = pd.DataFrame(data)
    
    # Tri par heure pour éviter les lignes croisées sur le graphique
    df['heure'] = df['heure'].astype(int)
    df = df.sort_values('heure').reset_index(drop=True)
    
    print(df.head(3))

    # --- GRAPHIC 1 : RENTABILITÉ HORAIRE ---
    plt.figure(figsize=(12, 5))
    sns.set_theme(style="whitegrid")
    
    # Tracé de la ligne de tendance
    plt.plot(df['heure'], df['gain_km'], marker='o', color='#e74c3c', linewidth=2.5, markersize=8, label='Gain moyen ($/km)')
    
    # Ajout d'une ligne horizontale pour la moyenne générale
    moyenne_globale = df['gain_km'].mean()
    plt.axhline(moyenne_globale, color='gray', linestyle='--', alpha=0.7, label=f"Moyenne ({moyenne_globale:.2f} $/km)")
    
    # Personnalisation des axes
    plt.title('Analyse de la Rentabilité Horaire (NYC Yellow Taxi)', fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Heure de la journée (Tranche horaire)', fontsize=11)
    plt.ylabel('Ratio : Recette / Distance ($/km)', fontsize=11)
    plt.xticks(range(0, 24))
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right')
    
    # Sauvegarde stricte SANS coupure des légendes (bbox_inches='tight')
    plt.savefig('../docs/images/rentabilite_horaire.png', dpi=300, bbox_inches='tight')
    plt.show() # Force l'affichage à l'écran dans le notebook
    plt.close() # Libère la mémoire pour éviter les chevauchements

    # --- GRAPHIC 2 : VOLUME DES COURSES ---
    plt.figure(figsize=(12, 5))
    sns.barplot(x='heure', y='nb_courses', data=df, color='#3498db', alpha=0.85)
    
    plt.title("Volume d'activité : Nombre total de courses traitées par heure", fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Heure de la journée', fontsize=11)
    plt.ylabel('Nombre de trajets', fontsize=11)
    plt.xticks(range(0, 24))
    
    plt.savefig('../docs/images/volume_activite.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print("✅ Les graphiques s'affichent maintenant correctement et sont enregistrés dans docs/images/ !")

ModuleNotFoundError: No module named 'pandas'